In [ ]:
from pathlib import Path
from io import BytesIO
import json, csv, hashlib, math

PACKAGE_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
                     if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError("Open this notebook from the package root or notebooks directory.")

SAVE_PDF = False

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import t
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from IPython.display import display, Image
plt.rcdefaults()

def load_npz(relative_path):
    with np.load(PACKAGE_ROOT / relative_path, allow_pickle=False) as archive:
        return {key: archive[key].copy() for key in archive.files}

def add_panel_labels(figure):
    panels = [axis for axis in figure.axes
              if axis.get_label() != "<colorbar>" and getattr(axis, "_colorbar", None) is None]
    assert len(panels) == 6, "Unexpected panel count"
    for index, axis in enumerate(panels):
        axis.text(-0.03, 1.04, f"({chr(97 + index)})",
                  transform=axis.transAxes, fontsize=12, fontweight="bold", fontfamily="DejaVu Serif",
                  ha="left", va="bottom", clip_on=False, gid="panel_label")
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    records = []
    for axis in panels:
        label = next(text for text in axis.texts if text.get_gid() == "panel_label")
        box = label.get_window_extent(renderer)
        for title in (title for panel in panels for title in (panel.title, panel._left_title, panel._right_title)):
            if title.get_text():
                assert not box.overlaps(title.get_window_extent(renderer)), "Panel label overlaps title"
        records.append({"label": label.get_text(), "axes_position": list(axis.get_position().bounds),
                        "label_bbox_pixels": list(box.bounds)})

def export_figure(figure, figure_id, tight=False):
    add_panel_labels(figure)
    options = {"bbox_inches": "tight"} if tight else {}
    preview = BytesIO()
    figure.savefig(preview, format="png", dpi=180, **options)
    display(Image(data=preview.getvalue()))
    if SAVE_PDF:
        output = PACKAGE_ROOT / "outputs"
        output.mkdir(exist_ok=True)
        figure.savefig(output / f"{figure_id}.pdf",
                       metadata={"CreationDate": None, "ModDate": None}, **options)
        print(f"Saved outputs/{figure_id}.pdf")


In [ ]:
metadata=json.loads((PACKAGE_ROOT/"data/FigS1_FigS2/metadata.json").read_text())
means={}
for key in metadata:
    values=load_npz(f"data/FigS1_FigS2/{key}.npz")["amplitude"]
    assert len(values)==metadata[key]["included_runs"]
    means[key]=values.mean(axis=0)
xf=np.arange(40)
freq_y_mean_noiseless=means["noiseless"]
freq_y_mean_noise=means["noise"]
freq_y_mean_noise2=means["noise2"]
freq_y_mean_noise3=means["noise3"]
freq_y_mean_noise4=means["noise4"]
freq_y_mean_noise5=means["noise5"]
print("Actual saved runs:",{key:rec["included_runs"] for key,rec in metadata.items()})

In [ ]:
from matplotlib import font_manager
font_manager.fontManager.addfont(str(PACKAGE_ROOT/"data/FigS1_FigS2/fonts/NimbusRoman-Regular.ttf"))
import matplotlib.cm as cm
plt.rcParams.update({"pdf.fonttype":42,"ps.fonttype":42,"font.family":"Nimbus Roman","font.size":14,"axes.linewidth":1.2})
def plot_fourier_spectra(xf,freq_y_mean_noise,freq_y_mean_noiseless,
                       freq_y_mean_noise2,freq_y_mean_noise3,
                       freq_y_mean_noise4,freq_y_mean_noise5):
    N_epoch = 500
    cmap = cm.get_cmap('viridis', N_epoch)
    norm = plt.Normalize(vmin=0, vmax=N_epoch)

    fig, axs = plt.subplots(2, 3, figsize=(12/1.1, 6*1.35), sharey=True)
    range_list = [wndex for wndex in range(0,200,10)] + \
                 [wndex for wndex in range(200,N_epoch,50)] + \
                 [N_epoch-1]

    for index in range_list:
        color = cmap(index)
        axs[0,1].plot(xf[1:26], freq_y_mean_noise[index][1:26], color=color,linewidth=0.7)
        axs[0,0].plot(xf[1:26], freq_y_mean_noiseless[index][1:26], color=color,linewidth=0.7)
        axs[1,0].plot(xf[1:26], freq_y_mean_noise2[index][1:26], color=color,linewidth=0.7)
        axs[0,2].plot(xf[1:26], freq_y_mean_noise3[index][1:26], color=color,linewidth=0.7)
        axs[1,1].plot(xf[1:26], freq_y_mean_noise4[index][1:26], color=color,linewidth=0.7)
        axs[1,2].plot(xf[1:26], freq_y_mean_noise5[index][1:26], color=color,linewidth=0.7)

    for ax in axs.ravel():
        ax.set_ylim(-0.001, 0.023)

    axs[1,0].set_xlabel(r"Frequency index: $\omega$")
    axs[1,1].set_xlabel(r"Frequency index: $\omega$")
    axs[1,2].set_xlabel(r"Frequency index: $\omega$")
    axs[0,0].set_ylabel("Amplitude")
    axs[1,0].set_ylabel("Amplitude")

    axs[0,0].set_title("Noiseless in rotation gate\nwith input $x$")
    axs[0,1].set_title("Dephasing noise:\n $p_x = p_y = 0, p_z = 0.15$")
    axs[0,2].set_title("Depolarizing noise:\n $p_x = p_y = p_z = 0.05$")
    axs[1,2].set_title("Noiseless in all gates\n")
    axs[1,0].set_title("Dephasing noise:\n $p_x = p_y = 0, p_z = 0.30$")
    axs[1,1].set_title("Depolarizing noise:\n $p_x = p_y = p_z = 0.025$")

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar_ax = fig.add_axes([1.01, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(sm, ax=axs.ravel().tolist(), 
                        ticks=np.linspace(0, N_epoch, 11, dtype=int),
                        orientation='vertical',cax=cbar_ax)
    cbar.set_label("Training Iteration", fontsize=12)
    cbar.ax.tick_params(labelsize=10)
    cbar.ax.tick_params(labelsize=12) 

    plt.tight_layout()
    return fig

In [ ]:
fig=plot_fourier_spectra(xf,freq_y_mean_noise,freq_y_mean_noiseless,freq_y_mean_noise2,freq_y_mean_noise3,freq_y_mean_noise4,freq_y_mean_noise5)
export_figure(fig,"FigS2",tight=True)
plt.close(fig)